# plot vertical profile of river tracer along the river channel

Builds a longitudinal "curtain" (fence diagram) of river-tracer concentration
beneath the Columbia River channel:

  x = distance downstream along the channel (km, from the MASS1 sections)
  y = depth below the riverbed (m, true layer thicknesses)
  color = river tracer concentration (-)

The mesh is a vertically extruded prism stack: ncells = nlayer * ele_per_layer,
with cell index = layer_index * ele_per_layer + column_index and layer 1 at the
TOP. Each MASS1 river section spans many columns, so the tracer is aggregated
across the channel width (default: max, matching plot_river_tracer.py).

Run on NERSC (default paths) or locally by setting USE_LOCAL = True.

Contact: Chao Zeng, chao.zeng@pnnl.gov

In [1]:
import os
import glob
from datetime import datetime, timedelta

import numpy as np
import h5py as h5

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

## I/O files and parameters

In [2]:
# --- I/O files (NERSC paths, following plot_simu_obs_wells-temp.py) ---

USE_LOCAL = False   # True -> use the copies under ../mesh/ on a local machine

work_dir = "/pscratch/sd/p/pshuai/HFR-riverbed"
src_dir = "/global/cfs/cdirs/m1800/pin/HFR-riverbed"
data_dir = "/global/cfs/cdirs/m1800/pin/github/HFR-thorne/data/"

# heterogeneous ("hete") case: heterogeneous permeability assigned to alluvium
case_name = "TH_perm_only_alluvium_2017-2019_daily-output_restart"
# case_name = "TH_perm_only_alluvium_2017-2019"           # hete, original run
# case_name = "TH_homo_alluvium_2017-2019"                # homogeneous case

model_dir = os.path.join(work_dir, case_name)
figure_dir = os.path.join(src_dir, "figures")

# PFLOTRAN writes <prefix>.h5, where the prefix comes from the input deck and is
# normally the case name. That is unverified for the *_restart run, so fall back
# to whatever single .h5 sits in model_dir if the expected name is absent.
fname_material_h5 = os.path.join(data_dir, "mesh", "hanford_mesh_100m-updated-matID-river.h5")
fname_pflotran_h5 = os.path.join(model_dir, case_name + ".h5")

if USE_LOCAL:
    fname_material_h5 = "../mesh/hanford_mesh_100m-updated-matID-river.h5"
    fname_pflotran_h5 = "../mesh/TH_perm_only_alluvium_2017-2019.h5"
    figure_dir = "../figures"

date_origin = datetime.strptime("2017-01-01 00:00:00", "%Y-%m-%d %H:%M:%S")

# key timestamps used throughout this project
target_dates = [
    "2017-06-10",   # high river stage
    "2017-08-29",   # high river temperature
    "2017-10-23",   # low river stage
    "2018-02-25",   # low river temperature
]

agg = "max"          # "max" (penetration envelope) or "mean" (bulk fraction)
depth_max = 100.0     # m below riverbed; tracer is identically zero below ~55 m
tracer_field = "Total Tracer_river [M]"

# period-wide scan over every output time step (2017-2019)
scan_all_times = True
penetration_threshold = 0.9   # tracer above this counts as river water

## Helper functions

Each function is independently testable: path resolution, the MASS1-section-to-column map, along-channel distance, layer thickness read from the mesh, time-group lookup, width aggregation, and the penetration front.

In [3]:
def resolve_pflotran_h5(path, model_dir):
    """Return an existing PFLOTRAN output file, globbing model_dir as a fallback."""
    if os.path.isfile(path):
        return path

    candidates = sorted(glob.glob(os.path.join(model_dir, "*.h5")))
    if not candidates:
        raise FileNotFoundError(
            f"No PFLOTRAN output found at {path} and no *.h5 in {model_dir}"
        )
    if len(candidates) > 1:
        print(f"WARNING: {len(candidates)} .h5 files in {model_dir}; using {candidates[0]}")
    print(f"NOTE: {path} not found, falling back to {candidates[0]}")
    return candidates[0]

In [4]:
def load_section_columns(material_h5):
    """Map each MASS1 river section to the mesh column indices it covers.

    Row i of Regions/Top is the top face of column i, so matching a section's
    faces against that table yields column indices directly.

    Returns (section_numbers, columns_per_section) ordered upstream -> downstream.
    """
    top_faces = material_h5["Regions"]["Top"]["Vertex Ids"][:, 1:4]
    face_to_column = {tuple(sorted(face)): i for i, face in enumerate(top_faces)}

    section_numbers = sorted(
        int(key.split("_")[1])
        for key in material_h5["Regions"].keys()
        if key.startswith("Mass1_")
    )

    columns_per_section = []
    n_unmatched = 0
    for isection in section_numbers:
        faces = material_h5["Regions"][f"Mass1_{isection}"]["Vertex Ids"][:, 1:4]
        columns = set()
        for face in faces:
            key = tuple(sorted(face))
            if key in face_to_column:
                columns.add(face_to_column[key])
            else:
                n_unmatched += 1
        columns_per_section.append(np.array(sorted(columns)))

    if n_unmatched:
        print(f"WARNING: {n_unmatched} MASS1 faces did not match a Top face")

    return section_numbers, columns_per_section

In [5]:
def section_distances(material_h5, columns_per_section):
    """Cumulative distance (km) downstream along the chain of section centroids.

    MASS1 section numbers increase downstream (riverbed drops from ~122 m to
    ~99 m between the first and last section), so the natural section order is
    already upstream -> downstream.
    """
    top_faces = material_h5["Regions"]["Top"]["Vertex Ids"][:, 1:4]
    vertices = material_h5["Domain"]["Vertices"][:]

    centroids = np.array([
        np.mean(vertices[top_faces[columns] - 1, :2], axis=(0, 1))
        for columns in columns_per_section
    ])

    steps = np.linalg.norm(np.diff(centroids, axis=0), axis=1)
    return np.concatenate([[0.0], np.cumsum(steps)]) / 1000.0

In [6]:
def layer_thickness(material_h5, ele_per_layer, n_layer):
    """Layer thicknesses (m, top layer first) read from the mesh geometry.

    Prism cells store the top face in columns 1:4 and the bottom face in 4:7.
    Every column shares the same thickness sequence, so column 0 is enough.
    """
    cells = material_h5["Domain"]["Cells"]
    vertices = material_h5["Domain"]["Vertices"][:]

    thickness = []
    for ilayer in range(n_layer):
        cell = cells[ilayer * ele_per_layer]
        z_top = vertices[cell[1:4] - 1, 2].mean()
        z_bot = vertices[cell[4:7] - 1, 2].mean()
        thickness.append(abs(z_top - z_bot))

    return np.array(thickness)

In [7]:
def list_time_groups(pflotran_h5, date_origin):
    """Return (group_names, datetimes) sorted in time.

    PFLOTRAN group names look like ' 152 Time  1.81208E+04 h'; the second-to-last
    token is the elapsed time in hours since date_origin.
    """
    groups = [key for key in pflotran_h5.keys() if "Time" in key]
    if not groups:
        raise KeyError("No 'Time' groups found in the PFLOTRAN output file")

    groups = sorted(groups, key=lambda name: float(name.split()[-2]))
    times = [date_origin + timedelta(hours=float(name.split()[-2])) for name in groups]

    return groups, times


def nearest_time_group(groups, times, target_date):
    """Pick the time group closest to target_date ('YYYY-MM-DD')."""
    target = datetime.strptime(target_date, "%Y-%m-%d")
    offsets = [abs((itime - target).total_seconds()) for itime in times]
    index = int(np.argmin(offsets))
    return groups[index], times[index]

In [8]:
def section_index(columns_per_section):
    """Flat gather index and segment offsets for reduceat-based aggregation.

    Concatenating every section's columns into one array lets the whole width
    aggregation run as a single ufunc.reduceat instead of a Python loop over
    sections - worth it when scanning ~1000 time steps.
    """
    counts = np.array([len(columns) for columns in columns_per_section])
    if np.any(counts == 0):
        raise ValueError("a MASS1 section has no columns; reduceat needs non-empty segments")

    flat_columns = np.concatenate(columns_per_section)
    offsets = np.concatenate([[0], np.cumsum(counts)[:-1]])

    return flat_columns, offsets, counts


def build_curtain(tracer, index, ele_per_layer, n_layer, agg="max"):
    """Collapse the 3D tracer field to a (n_section, n_layer) curtain.

    Cell index = layer * ele_per_layer + column, so a single reshape gives a
    (layer, column) view; the section gather and reduction then vectorize.
    """
    flat_columns, offsets, counts = index
    gathered = tracer.reshape(n_layer, ele_per_layer)[:, flat_columns]

    if agg == "max":
        reduced = np.maximum.reduceat(gathered, offsets, axis=1)
    elif agg == "mean":
        reduced = np.add.reduceat(gathered, offsets, axis=1) / counts
    else:
        raise ValueError(f"agg must be 'max' or 'mean', got {agg!r}")

    return reduced.T


def penetration_depth(curtain, depth_edge_full, threshold):
    """Depth (m) of the deepest layer whose value exceeds threshold, per section.

    Uses the deepest exceeding layer rather than a contiguous front, so isolated
    deep cells along preferential flow paths are kept rather than clipped at the
    first gap. A consequence of that choice: taking the period maximum of the
    per-snapshot depth gives the same answer as taking the depth of the
    period-maximum curtain, so a single accumulator is enough.

    Sections that never exceed the threshold return 0.
    """
    above = curtain > threshold
    n_layer = curtain.shape[1]

    # argmax on the reversed axis finds the last True
    deepest = n_layer - 1 - np.argmax(above[:, ::-1], axis=1)

    return np.where(above.any(axis=1), depth_edge_full[deepest + 1], 0.0)

In [9]:
def cell_edges(centers):
    """Midpoint edges for pcolormesh, extrapolating half a cell at each end."""
    inner = 0.5 * (centers[1:] + centers[:-1])
    first = centers[0] - (inner[0] - centers[0])
    last = centers[-1] + (centers[-1] - inner[-1])
    return np.concatenate([[first], inner, [last]])


def draw_curtain(ax, x_edges, depth_edge, curtain, cmap, norm):
    """Draw one curtain panel; returns the QuadMesh."""
    mesh = ax.pcolormesh(
        x_edges, depth_edge, curtain.T,
        cmap=cmap, norm=norm, shading="flat",
    )
    ax.set_ylim(depth_edge[-1], 0)   # depth increases downward
    ax.set_xlim(x_edges[0], x_edges[-1])
    return mesh

## Load mesh geometry

The mesh is an extruded prism stack, so cell index = `layer * ele_per_layer + column`, with **layer 1 at the top**.

In [10]:
# --- load mesh geometry ---

material_h5 = h5.File(fname_material_h5, "r")

layer_ids = material_h5["Materials"]["Layer Ids"][:]
n_layer = len(np.unique(layer_ids))
ele_per_layer = int(np.sum(layer_ids == np.unique(layer_ids)[0]))
n_cell = n_layer * ele_per_layer

print(f"mesh: {n_cell} cells = {n_layer} layers x {ele_per_layer} columns")

section_numbers, columns_per_section = load_section_columns(material_h5)
distance_km = section_distances(material_h5, columns_per_section)
thickness = layer_thickness(material_h5, ele_per_layer, n_layer)
index = section_index(columns_per_section)

material_h5.close()

widths = np.array([len(c) for c in columns_per_section])
print(f"MASS1 sections: {len(section_numbers)} "
      f"(Mass1_{section_numbers[0]} to Mass1_{section_numbers[-1]}), "
      f"{widths.min()}-{widths.max()} columns each")
print(f"channel length: {distance_km[-1]:.1f} km")
print(f"layer thickness: {thickness[:6].round(2).tolist()} ... total {thickness.sum():.1f} m")

mesh: 902937 cells = 31 layers x 29127 columns
MASS1 sections: 325 (Mass1_35 to Mass1_359), 2-126 columns each
channel length: 83.4 km
layer thickness: [0.1, 0.15, 0.22, 0.34, 0.51, 0.68] ... total 100.0 m


In [11]:
# --- vertical axis: layer interface depths, clipped to depth_max ---

depth_edge_full = np.concatenate([[0.0], np.cumsum(thickness)])
n_layer_plot = int(np.searchsorted(depth_edge_full, depth_max, side="left"))
depth_edge = depth_edge_full[: n_layer_plot + 1]

x_edges = cell_edges(distance_km)

print(f"plotting top {n_layer_plot} layers, 0 to {depth_edge[-1]:.1f} m depth")

plotting top 31 layers, 0 to 100.0 m depth


## Tracer snapshots at the key dates

In [12]:
# --- load tracer snapshots at the target dates ---

fname_pflotran_h5 = resolve_pflotran_h5(fname_pflotran_h5, model_dir)
pflotran_h5 = h5.File(fname_pflotran_h5, "r")

groups, times = list_time_groups(pflotran_h5, date_origin)
print(f"output: {os.path.basename(fname_pflotran_h5)}, {len(groups)} time steps "
      f"({times[0]:%Y-%m-%d} to {times[-1]:%Y-%m-%d})")

curtains = []
for target_date in target_dates:
    group, itime = nearest_time_group(groups, times, target_date)

    tracer = np.asarray(pflotran_h5[group][tracer_field])
    if tracer.size != n_cell:
        raise ValueError(
            f"tracer has {tracer.size} values but the mesh has {n_cell} cells"
        )

    curtain = build_curtain(tracer, index, ele_per_layer, n_layer, agg)
    curtains.append((target_date, itime, curtain[:, :n_layer_plot]))

    note = "" if itime.date() == datetime.strptime(target_date, "%Y-%m-%d").date() \
        else f"  (nearest available to {target_date})"
    print(f"  {itime:%Y-%m-%d}: tracer {curtain.min():.3f}-{curtain.max():.3f}{note}")

pflotran_h5.close()

NOTE: /pscratch/sd/p/pshuai/HFR-riverbed/TH_perm_only_alluvium_2017-2019_daily-output_restart/TH_perm_only_alluvium_2017-2019_daily-output_restart.h5 not found, falling back to /pscratch/sd/p/pshuai/HFR-riverbed/TH_perm_only_alluvium_2017-2019_daily-output_restart/TH_perm_only_alluvium_2017-2019_daily-output-10080.0000h.h5


KeyError: "No 'Time' groups found in the PFLOTRAN output file"

## Figures: one curtain per key date

In [13]:
# --- multi-panel figure: one curtain per key date ---

cmap = plt.get_cmap("Spectral_r")
norm = Normalize(vmin=0, vmax=1)     # clips the ~1.001 numerical overshoot

labels = ["high river stage", "high river temperature",
          "low river stage", "low river temperature"]

fig, axes = plt.subplots(
    len(curtains), 1, figsize=(10, 2.4 * len(curtains)),
    sharex=True, constrained_layout=True,
)
axes = np.atleast_1d(axes)

for ax, (target_date, itime, curtain), label in zip(axes, curtains, labels):
    draw_curtain(ax, x_edges, depth_edge, curtain, cmap, norm)
    ax.set_ylabel("Depth (m)", fontsize=11)
    ax.set_title(f"{itime:%Y-%m-%d} - {label}", fontsize=11, loc="left")

axes[-1].set_xlabel("Distance downstream along channel (km)", fontsize=12)

cbar = fig.colorbar(
    ScalarMappable(norm=norm, cmap=cmap), ax=axes.tolist(),
    orientation="vertical", extend="neither", shrink=0.9, aspect=30,
)
cbar.ax.set_ylabel(f"River tracer ({agg} across channel) (-)",
                   rotation=270, labelpad=20, fontsize=11)

fig.suptitle("River tracer beneath the channel - heterogeneous permeability",
             fontsize=13)

os.makedirs(figure_dir, exist_ok=True)
fname_fig = os.path.join(figure_dir, f"tracer_curtain_along_channel_{agg}.jpg")
fig.savefig(fname_fig, dpi=300, bbox_inches="tight")
print(f"saved {fname_fig}")
plt.show()

NameError: name 'curtains' is not defined

In [ ]:
# --- single-date figures ---

for target_date, itime, curtain in curtains:
    fig_one, ax_one = plt.subplots(figsize=(10, 3.2), constrained_layout=True)

    mesh = draw_curtain(ax_one, x_edges, depth_edge, curtain, cmap, norm)

    ax_one.set_xlabel("Distance downstream along channel (km)", fontsize=12)
    ax_one.set_ylabel("Depth below riverbed (m)", fontsize=12)
    ax_one.set_title(f"River tracer along the channel, {itime:%Y-%m-%d} "
                     f"(hete, {agg} across channel)", fontsize=12)

    cbar_one = fig_one.colorbar(mesh, ax=ax_one, orientation="vertical",
                               shrink=0.9, aspect=20)
    cbar_one.ax.set_ylabel("River tracer (-)", rotation=270, labelpad=20, fontsize=11)

    fname_one = os.path.join(
        figure_dir, f"tracer_curtain_{itime:%Y-%m-%d}_{agg}.jpg"
    )
    fig_one.savefig(fname_one, dpi=300, bbox_inches="tight")
    print(f"saved {fname_one}")
    plt.close(fig_one)

## Period-wide maximum over the full record

A single streaming pass over every output time step. The accumulator is the `(n_section, n_layer)` curtain, not the full field, so memory is independent of record length.

In [ ]:
# --- period-wide scan: max tracer per cell over the full 2017-2019 record ---
#
# One streaming pass over every output time step. The accumulator is the
# (n_section, n_layer) curtain rather than the 902,937-cell field, so memory
# stays negligible regardless of how many time steps the run wrote.

curtain_period_max = None
front_depth = None

if scan_all_times:
    pflotran_h5 = h5.File(fname_pflotran_h5, "r")
    groups, times = list_time_groups(pflotran_h5, date_origin)

    print(f"scanning {len(groups)} time steps "
          f"({times[0]:%Y-%m-%d} to {times[-1]:%Y-%m-%d}) ...")

    curtain_period_max = np.zeros((len(section_numbers), n_layer))
    for istep, group in enumerate(groups):
        tracer = np.asarray(pflotran_h5[group][tracer_field])
        curtain_period_max = np.maximum(
            curtain_period_max,
            build_curtain(tracer, index, ele_per_layer, n_layer, agg),
        )
        if (istep + 1) % 100 == 0 or istep + 1 == len(groups):
            print(f"  {istep + 1}/{len(groups)} steps")

    pflotran_h5.close()

    curtain_period_max = curtain_period_max[:, :n_layer_plot]

    front_depth = penetration_depth(
        curtain_period_max, depth_edge_full, penetration_threshold
    )

    reached = front_depth > 0
    print(f"period max tracer: {curtain_period_max.max():.3f}")
    print(f"penetration depth (tracer > {penetration_threshold}): "
          f"{front_depth[reached].min():.1f}-{front_depth.max():.1f} m, "
          f"median {np.median(front_depth[reached]):.1f} m, "
          f"{np.sum(~reached)} of {len(front_depth)} sections never exceed it")

In [ ]:
# --- two-panel figure: period-max curtain, with the penetration front below ---

if curtain_period_max is not None:
    fig_max, (ax_curtain, ax_front) = plt.subplots(
        2, 1, figsize=(10, 6), sharex=True,
        gridspec_kw={"height_ratios": [3, 1.4]}, constrained_layout=True,
    )

    mesh_max = draw_curtain(
        ax_curtain, x_edges, depth_edge, curtain_period_max, cmap, norm
    )
    ax_curtain.plot(distance_km, front_depth, color="black", lw=1.0,
                    label=f"tracer > {penetration_threshold}")
    ax_curtain.set_ylabel("Depth below riverbed (m)", fontsize=12)
    ax_curtain.set_title(
        f"Maximum river tracer over {times[0]:%Y-%m-%d} to {times[-1]:%Y-%m-%d} "
        f"(hete, {agg} across channel)", fontsize=12,
    )
    ax_curtain.legend(loc="lower right", fontsize=9, framealpha=0.9)

    # attach to both axes so constrained_layout reserves width from each and the
    # two panels stay horizontally aligned
    cbar_max = fig_max.colorbar(mesh_max, ax=[ax_curtain, ax_front],
                                orientation="vertical", shrink=0.9, aspect=30)
    cbar_max.ax.set_ylabel("Max river tracer (-)", rotation=270,
                           labelpad=20, fontsize=11)

    ax_front.fill_between(distance_km, front_depth, 0,
                          color="steelblue", alpha=0.35)
    ax_front.plot(distance_km, front_depth, color="steelblue", lw=1.0)
    ax_front.set_ylim(depth_edge[-1], 0)
    ax_front.set_xlim(x_edges[0], x_edges[-1])
    ax_front.set_xlabel("Distance downstream along channel (km)", fontsize=12)
    ax_front.set_ylabel("Max penetration\ndepth (m)", fontsize=11)
    ax_front.grid(alpha=0.3)

    fname_max = os.path.join(
        figure_dir, f"tracer_curtain_period_max_{agg}.jpg"
    )
    fig_max.savefig(fname_max, dpi=300, bbox_inches="tight")
    print(f"saved {fname_max}")
    plt.show()